# 🏥 Healthcare Q&A Chatbot
**Powered by Anthropic Claude API**

This chatbot is designed to help patients and users get reliable, easy-to-understand answers to general healthcare questions. It addresses the gap in healthcare information accessibility — a real-world problem observed through hands-on experience in clinical settings.

> ⚠️ **Disclaimer:** This chatbot is for informational purposes only and does not constitute medical advice. Always consult a licensed healthcare professional for diagnosis and treatment.

---
### Features
- 💬 Multi-turn conversation with memory
- 🏥 Healthcare-specialized system prompt
- 🌐 Bilingual support (English & Korean)
- 🔒 Safe response guardrails
- 📋 Conversation history export

## 1. Installation
Run the cell below to install required packages.

In [ ]:
# Install required packages
!pip install anthropic python-dotenv

## 2. Setup & Configuration

In [ ]:
import anthropic
import os
import json
from datetime import datetime
from dotenv import load_dotenv

# Load API key from .env file (recommended) or set directly
load_dotenv()

# Option 1: Use .env file (create a .env file with: ANTHROPIC_API_KEY=your_key_here)
# Option 2: Set directly (not recommended for production)
# os.environ["ANTHROPIC_API_KEY"] = "your_api_key_here"

# Initialize Anthropic client
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("✅ Anthropic client initialized successfully!")

## 3. System Prompt — Healthcare Specialist Configuration

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable and empathetic Healthcare Q&A Assistant. 
Your role is to help patients and users understand general health-related topics 
in a clear, accessible, and compassionate manner.

## Your Responsibilities:
- Answer general healthcare questions about symptoms, conditions, medications, and wellness
- Explain medical terminology in simple, easy-to-understand language
- Provide information about when to seek professional medical attention
- Guide users on navigating healthcare services (e.g., what type of specialist to see)
- Support both English and Korean language queries

## Important Guidelines:
- ALWAYS remind users that your responses are for informational purposes only
- NEVER provide a definitive diagnosis or prescribe medication
- If a user describes a medical emergency (e.g., chest pain, difficulty breathing, stroke symptoms), 
  IMMEDIATELY advise them to call emergency services (119 in Korea, 911 in the US)
- Be empathetic and patient — users may be anxious or confused about their health
- If asked about eye/vision health, provide especially detailed and careful responses

## Response Format:
- Keep responses concise but thorough (aim for 150-300 words)
- Use bullet points for lists of symptoms or steps
- End responses that involve symptoms with a recommendation to consult a healthcare provider
- Match the language of the user (respond in Korean if the user writes in Korean)

Remember: You bridge the gap between patients and the healthcare system by making 
information accessible and reducing health anxiety through clear communication.
"""

print("✅ System prompt configured!")

## 4. Chatbot Core Functions

In [ ]:
class HealthcareChatbot:
    """
    A multi-turn Healthcare Q&A chatbot powered by Anthropic Claude.
    Maintains conversation history for contextual responses.
    """

    def __init__(self, model: str = "claude-sonnet-4-20250514"):
        self.client = client
        self.model = model
        self.conversation_history = []
        self.session_start = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"🏥 Healthcare Chatbot initialized | Session started: {self.session_start}")
        print(f"🤖 Model: {self.model}")
        print("-" * 60)

    def chat(self, user_message: str) -> str:
        """
        Send a message and receive a response.
        Maintains full conversation history for context.
        """
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })

        # Call Claude API
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=self.conversation_history
        )

        # Extract response text
        assistant_message = response.content[0].text

        # Add assistant response to history
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })

        return assistant_message

    def reset_conversation(self):
        """Clear conversation history to start a new session."""
        self.conversation_history = []
        self.session_start = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print("🔄 Conversation reset. New session started!")

    def export_conversation(self, filename: str = None):
        """Export conversation history to a JSON file."""
        if not filename:
            filename = f"conversation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        export_data = {
            "session_start": self.session_start,
            "model": self.model,
            "conversation": self.conversation_history
        }

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(export_data, f, ensure_ascii=False, indent=2)

        print(f"💾 Conversation exported to: {filename}")
        return filename

    def show_history(self):
        """Display the full conversation history."""
        if not self.conversation_history:
            print("No conversation history yet.")
            return
        print("\n" + "=" * 60)
        print("📋 CONVERSATION HISTORY")
        print("=" * 60)
        for i, msg in enumerate(self.conversation_history):
            role = "👤 You" if msg["role"] == "user" else "🏥 Assistant"
            print(f"\n{role}:")
            print(msg["content"])
            print("-" * 60)

print("✅ HealthcareChatbot class defined!")

## 5. Interactive Chat Interface

In [ ]:
def run_chat_interface():
    """
    Run an interactive chat session in the notebook.
    Type 'quit' or 'exit' to end, 'reset' to start over, 
    'history' to see full chat, 'export' to save.
    """
    bot = HealthcareChatbot()

    print("\n🏥 Welcome to the Healthcare Q&A Chatbot!")
    print("건강 관련 질문을 자유롭게 물어보세요. (Korean is also supported!)")
    print("\nCommands: 'quit' | 'reset' | 'history' | 'export'")
    print("=" * 60)

    while True:
        try:
            user_input = input("\n👤 You: ").strip()

            if not user_input:
                continue

            # Handle commands
            if user_input.lower() in ["quit", "exit", "종료"]:
                print("\n👋 Thank you for using the Healthcare Chatbot. Stay healthy!")
                break
            elif user_input.lower() == "reset":
                bot.reset_conversation()
                continue
            elif user_input.lower() == "history":
                bot.show_history()
                continue
            elif user_input.lower() == "export":
                bot.export_conversation()
                continue

            # Get response
            print("\n🏥 Assistant: ", end="")
            response = bot.chat(user_input)
            print(response)

        except KeyboardInterrupt:
            print("\n\n👋 Session ended. Stay healthy!")
            break

    return bot

# Start chatting!
bot = run_chat_interface()

## 6. Example Usage (Demo without interactive input)

In [ ]:
# Demo: Run example questions automatically
demo_bot = HealthcareChatbot()

example_questions = [
    "What are the common symptoms of dry eye syndrome?",
    "How can I tell if I need to see a doctor for my eye redness?",
    "눈이 자주 충혈되는데, 어떤 이유가 있을까요?",  # Korean: Why do my eyes get red often?
]

for question in example_questions:
    print(f"\n👤 You: {question}")
    print("\n🏥 Assistant:")
    response = demo_bot.chat(question)
    print(response)
    print("\n" + "=" * 60)

## 7. Export Conversation

In [ ]:
# Export the demo conversation to JSON
demo_bot.export_conversation("demo_conversation.json")

# View conversation history
demo_bot.show_history()